# 04 — Final Model Test Notebook

Bu notebook `best_model_final.keras` modelini gerçek test videoları üzerinde test eder.

- Test klasörü: `/content/drive/MyDrive/sign-language-project/dataset/test`
- Kullanılan model: `best_model_final.keras`
- Kullanılan asset klasörü: `demo_assets_final`
- Kullanılacak videolar: sadece `*_color.mp4`
- Kullanılmayacak videolar: `*_depth.mp4`
- Preprocessing: `color_only + relative_coords + finger_angles + zscore`
- Model input beklenen shape: `(16, 156)`

In [ ]:
# ============================================================
# 1) Install
# ============================================================
!pip install -q mediapipe==0.10.21 opencv-python-headless scikit-learn tqdm
print("Kurulum tamam.")

In [ ]:
# ============================================================
# 2) Imports + Drive
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, json, time, pickle, traceback
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
import tensorflow as tf
from tensorflow import keras
from tqdm.notebook import tqdm

from sklearn.metrics import (
    accuracy_score,
    top_k_accuracy_score,
    classification_report,
    confusion_matrix
)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================================
# 3) Paths / Config
# ============================================================
BASE = Path("/content/drive/MyDrive/sign-language-project")
DATASET = BASE / "dataset"
TEST_DIR = DATASET / "test"

MODEL_PATH = BASE / "best_model_final.keras"
DEMO_DIR = BASE / "demo_assets_final"

TEST_CACHE = DATASET / "cached_landmarks" / "test_color_final"
PACKED_DIR = BASE / "packed_landmarks"
RESULTS_DIR = BASE / "test_results_final"

TEST_CACHE.mkdir(parents=True, exist_ok=True)
PACKED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 16
RAW_FEAT_DIM = 126     # color only: left_hand(63) + right_hand(63)
FINAL_FEAT_DIM = 156   # 126 + finger_angles(30)

# Cache yeniden oluşturulsun mu?
# False: varsa eski landmark/paketleri kullanır.
# True: test cache ve packed test dosyalarını tekrar üretir.
FORCE_REBUILD = False

print("BASE:", BASE, BASE.exists())
print("TEST_DIR:", TEST_DIR, TEST_DIR.exists())
print("MODEL_PATH:", MODEL_PATH, MODEL_PATH.exists())
print("DEMO_DIR:", DEMO_DIR, DEMO_DIR.exists())
print("TEST_CACHE:", TEST_CACHE)
print("PACKED_DIR:", PACKED_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

assert BASE.exists(), "BASE bulunamadı."
assert TEST_DIR.exists(), "dataset/test klasörü bulunamadı."
assert MODEL_PATH.exists(), "best_model_final.keras bulunamadı."
assert DEMO_DIR.exists(), "demo_assets_final klasörü bulunamadı."

In [ ]:
# ============================================================
# 4) Test klasörünü kontrol et
# ============================================================
all_mp4 = sorted(TEST_DIR.glob("*.mp4"))
color_videos = sorted(TEST_DIR.glob("*_color.mp4"))
depth_videos = sorted(TEST_DIR.glob("*_depth.mp4"))

print("Toplam mp4:", len(all_mp4))
print("Color video:", len(color_videos))
print("Depth video:", len(depth_videos))

print("\nİlk 10 color video:")
for p in color_videos[:10]:
    print(" -", p.name)

print("\nİlk 10 depth video:")
for p in depth_videos[:10]:
    print(" -", p.name)

if len(color_videos) == 0:
    raise RuntimeError("Test klasöründe *_color.mp4 video bulunamadı.")

In [ ]:
# ============================================================
# 5) Label CSV dosyasını otomatik bul ve oku
# ============================================================
def read_label_file(path):
    path = Path(path)

    # Önce header'sız oku: AUTSL benzeri dosyalar genelde sample_id,label şeklinde olur.
    df0 = pd.read_csv(path, header=None)

    # Eğer ilk satır header gibi görünüyorsa header=0 ile tekrar oku.
    first_row = [str(x).lower() for x in df0.iloc[0].tolist()]
    header_keywords = ["sample", "id", "label", "class", "classid", "class_id"]
    looks_like_header = any(any(k in cell for k in header_keywords) for cell in first_row)

    if looks_like_header:
        df = pd.read_csv(path)

        sample_col = None
        label_col = None

        for c in df.columns:
            cl = c.lower()
            if sample_col is None and ("sample" in cl or "video" in cl or cl in ["id", "file", "filename"]):
                sample_col = c
            if label_col is None and ("label" in cl or "class" in cl):
                label_col = c

        if sample_col is None:
            sample_col = df.columns[0]
        if label_col is None:
            label_col = df.columns[1]

        df = df[[sample_col, label_col]].copy()
        df.columns = ["sample_id", "label"]

    else:
        if df0.shape[1] < 2:
            raise ValueError(f"{path.name} en az 2 kolon içermeli: sample_id,label")

        df = df0.iloc[:, :2].copy()
        df.columns = ["sample_id", "label"]

    df["sample_id"] = df["sample_id"].astype(str)

    # sample_id yanlışlıkla dosya adı olarak gelirse temizle
    df["sample_id"] = (
        df["sample_id"]
        .str.replace(".mp4", "", regex=False)
        .str.replace("_color", "", regex=False)
        .str.replace("_depth", "", regex=False)
    )

    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)

    return df


possible_label_files = [
    DATASET / "test_labels.csv",
    DATASET / "Test_labels.csv",
    DATASET / "test.csv",
    DATASET / "test_labels.txt",
    DATASET / "test.txt",
]

# dataset içinde test geçen csv/txt dosyalarını da ekle
for p in sorted(DATASET.glob("*")):
    if p.suffix.lower() in [".csv", ".txt"] and "test" in p.name.lower():
        if p not in possible_label_files:
            possible_label_files.append(p)

TEST_LABEL_FILE = None
for p in possible_label_files:
    if p.exists():
        TEST_LABEL_FILE = p
        break

print("Bulunan TEST_LABEL_FILE:", TEST_LABEL_FILE)

if TEST_LABEL_FILE is not None:
    test_df = read_label_file(TEST_LABEL_FILE)
    print("Test label satır sayısı:", len(test_df))
    print("Unique original label:", test_df["label"].nunique())
    print("Label range:", test_df["label"].min(), "-", test_df["label"].max())
    display(test_df.head())

else:
    print("UYARI: Test label dosyası bulunamadı.")
    print("Accuracy hesaplanamaz; sadece prediction CSV üretilecek.")

    # Label yoksa sample_id listesini color videolardan çıkar.
    sample_ids_auto = [p.stem.replace("_color", "") for p in color_videos]
    test_df = pd.DataFrame({"sample_id": sample_ids_auto, "label": -1})
    display(test_df.head())

In [ ]:
# ============================================================
# 6) Model + final demo assetlerini yükle
# ============================================================
model = keras.models.load_model(MODEL_PATH, compile=False)

print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)

# Final model assetleri
norm_path = DEMO_DIR / "norm_stats.json"
classes_path = DEMO_DIR / "label_encoder_classes.npy"
label_map_path = DEMO_DIR / "label_map.json"
config_path = DEMO_DIR / "demo_config.json"

assert norm_path.exists(), "demo_assets_final/norm_stats.json bulunamadı."
assert classes_path.exists(), "demo_assets_final/label_encoder_classes.npy bulunamadı."
assert label_map_path.exists(), "demo_assets_final/label_map.json bulunamadı."

with open(norm_path, "r") as f:
    norm_stats = json.load(f)

feat_mean = np.array(norm_stats["mean"], dtype=np.float32)
feat_std = np.array(norm_stats["std"], dtype=np.float32)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)

label_encoder_classes = np.load(classes_path).astype(int)

with open(label_map_path, "r", encoding="utf-8") as f:
    label_map_raw = json.load(f)

# JSON keyleri string olabilir; int'e çeviriyoruz.
label_map = {int(k): v for k, v in label_map_raw.items()}

if config_path.exists():
    with open(config_path, "r") as f:
        demo_config = json.load(f)
    print("\nDemo config:")
    print(json.dumps(demo_config, indent=2, ensure_ascii=False))

print("\nNorm mean shape:", feat_mean.shape)
print("Norm std shape:", feat_std.shape)
print("Label encoder classes:", label_encoder_classes.shape)
print("First 10 original classes:", label_encoder_classes[:10])

assert feat_mean.shape[0] == FINAL_FEAT_DIM, f"norm_stats feature dim {feat_mean.shape[0]}, beklenen {FINAL_FEAT_DIM}"
assert feat_std.shape[0] == FINAL_FEAT_DIM, f"norm_stats feature dim {feat_std.shape[0]}, beklenen {FINAL_FEAT_DIM}"
assert model.input_shape[-1] == FINAL_FEAT_DIM, f"model input feat dim {model.input_shape[-1]}, beklenen {FINAL_FEAT_DIM}"
assert model.input_shape[1] == SEQ_LEN, f"model seq len {model.input_shape[1]}, beklenen {SEQ_LEN}"
assert model.output_shape[-1] == len(label_encoder_classes), "Model output ile label_encoder_classes uyuşmuyor."

In [ ]:
# ============================================================
# 7) Final modelin bildiği sınıfları filtrele
# ============================================================
known_original_classes = set(label_encoder_classes.tolist())
orig_to_new = {int(orig): int(i) for i, orig in enumerate(label_encoder_classes.tolist())}
new_to_orig = {int(i): int(orig) for i, orig in enumerate(label_encoder_classes.tolist())}

if (test_df["label"] >= 0).any():
    test_df["is_known_by_model"] = test_df["label"].isin(known_original_classes)

    known_test_df = test_df[test_df["is_known_by_model"]].copy()
    unknown_test_df = test_df[~test_df["is_known_by_model"]].copy()

    print("Toplam test sample:", len(test_df))
    print("Modelin bildiği sınıflardaki test sample:", len(known_test_df))
    print("Model dışında kalan test sample:", len(unknown_test_df))

    if len(unknown_test_df) > 0:
        print("\nModelin bilmediği original class_id örnekleri:")
        print(sorted(unknown_test_df["label"].unique())[:100])

else:
    known_test_df = test_df.copy()
    unknown_test_df = pd.DataFrame()
    print("Label olmadığı için tüm color videolar prediction için kullanılacak.")

display(known_test_df.head())

In [ ]:
# ============================================================
# 8) MediaPipe Hands: sadece color video -> 126 landmark
# ============================================================
mp_hands = mp.solutions.hands

def hands_to_vec(results):
    """
    MediaPipe Hands çıktısından 126 boyutlu vektör çıkarır.
    Left hand: 63, Right hand: 63
    Eksik el: 0
    """
    left = [0.0] * 63
    right = [0.0] * 63

    if results.multi_hand_landmarks and results.multi_handedness:
        for hand_lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
            label = handedness.classification[0].label

            vec = []
            for lm in hand_lm.landmark:
                vec.extend([lm.x, lm.y, lm.z])

            if label == "Left":
                left = vec
            else:
                right = vec

    return np.array(left + right, dtype=np.float32)


def find_color_video(sample_id):
    """
    CSV sample_id -> gerçek color video path
    Öncelik: sample_id_color.mp4
    """
    sid = str(sample_id)
    candidates = [
        TEST_DIR / f"{sid}_color.mp4",
        TEST_DIR / f"{sid}.mp4",
    ]

    # Eğer sample_id içinde extension veya _color varsa
    clean = sid.replace(".mp4", "").replace("_color", "").replace("_depth", "")
    candidates += [
        TEST_DIR / f"{clean}_color.mp4",
        TEST_DIR / f"{clean}.mp4",
    ]

    for p in candidates:
        if p.exists():
            return p

    return None


def video_to_landmarks_color_only(video_path, max_frames=SEQ_LEN):
    """
    Bir color videodan 16 frame seçip her frame için hand landmark çıkarır.
    Output: (16, 126)
    """
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return np.zeros((max_frames, RAW_FEAT_DIM), dtype=np.float32)

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()
        return np.zeros((max_frames, RAW_FEAT_DIM), dtype=np.float32)

    target_indices = set(np.linspace(0, total - 1, num=max_frames, dtype=int).tolist())
    grabbed = {}
    fid = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if fid in target_indices:
            grabbed[fid] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        fid += 1

    cap.release()

    frames = [
        grabbed.get(i, np.zeros((256, 256, 3), dtype=np.uint8))
        for i in sorted(target_indices)
    ]

    landmarks = []

    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=2,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as hands:
        for frame in frames:
            results = hands.process(frame)
            landmarks.append(hands_to_vec(results))

    while len(landmarks) < max_frames:
        landmarks.append(
            landmarks[-1].copy() if landmarks else np.zeros(RAW_FEAT_DIM, dtype=np.float32)
        )

    return np.array(landmarks[:max_frames], dtype=np.float32)


# Hızlı kontrol
example_sid = str(known_test_df.iloc[0]["sample_id"])
example_video = find_color_video(example_sid)
print("Example sample_id:", example_sid)
print("Example color video:", example_video)

if example_video is None:
    print("UYARI: İlk örneğin color videosu bulunamadı.")
else:
    arr = video_to_landmarks_color_only(example_video)
    print("Example landmark shape:", arr.shape)
    print("Nonzero count:", np.count_nonzero(arr))

In [ ]:
# ============================================================
# 9) Test landmark extraction
# ============================================================
def safe_cache_name(sample_id):
    return str(sample_id).replace("/", "_").replace("\\", "_").replace(":", "_")


def run_test_extraction(df):
    done = 0
    skipped = 0
    missing_color = 0
    errors = 0

    start = time.time()

    for _, row in tqdm(df.iterrows(), total=len(df), desc="TEST color landmark extraction"):
        sid = str(row["sample_id"])
        cache_path = TEST_CACHE / f"{safe_cache_name(sid)}.npy"

        if cache_path.exists() and not FORCE_REBUILD:
            skipped += 1
            continue

        color_path = find_color_video(sid)

        if color_path is None:
            arr = np.zeros((SEQ_LEN, RAW_FEAT_DIM), dtype=np.float32)
            np.save(cache_path, arr)
            missing_color += 1
            continue

        try:
            arr = video_to_landmarks_color_only(color_path)

            if arr.shape != (SEQ_LEN, RAW_FEAT_DIM):
                print("Shape problem:", sid, arr.shape)
                arr = np.zeros((SEQ_LEN, RAW_FEAT_DIM), dtype=np.float32)

            np.save(cache_path, arr)
            done += 1

        except Exception as e:
            print("Hata:", sid, e)
            traceback.print_exc()
            arr = np.zeros((SEQ_LEN, RAW_FEAT_DIM), dtype=np.float32)
            np.save(cache_path, arr)
            errors += 1

    elapsed = time.time() - start

    print("\nExtraction tamamlandı")
    print("Yeni işlenen:", done)
    print("Atlanan:", skipped)
    print("Color video eksik:", missing_color)
    print("Hata:", errors)
    print("Süre:", f"{elapsed/60:.1f} dk")


run_test_extraction(known_test_df)

In [ ]:
# ============================================================
# 10) Preprocessing: relative coords + finger angles + zscore
#     Eğitimdeki final model preprocessing ile aynıdır.
# ============================================================
def normalize_hands_relative(X):
    """
    Input:  (N, 16, 126)
    Output: (N, 16, 126)

    Her el için wrist landmarkını çıkarır ve landmark 9 uzaklığına göre scale eder.
    Eğitim notebookundaki final model preprocessing ile aynı mantık.
    """
    X = X.astype(np.float32)
    result = X.copy()
    N, T, D = result.shape

    assert T == SEQ_LEN
    assert D == RAW_FEAT_DIM

    for start in [0, 63]:
        hand = result[:, :, start:start+63].reshape(N, T, 21, 3)
        wrist = hand[:, :, 0:1, :]
        hand_rel = hand - wrist

        scale_vec = hand_rel[:, :, 9, :]
        scale = np.linalg.norm(scale_vec, axis=-1, keepdims=True)[:, :, :, np.newaxis]
        scale = np.where(scale < 1e-6, 1.0, scale)

        result[:, :, start:start+63] = (hand_rel / scale).reshape(N, T, 63)

    return result


def compute_finger_angles(X):
    """
    Input:  (N, 16, 126)
    Output: (N, 16, 156)

    Her el için 15 açı çıkarır: 2 el x 15 = 30 açı.
    126 landmark feature + 30 angle feature = 156 feature.
    """
    X = X.astype(np.float32)
    N, T, D = X.shape

    assert T == SEQ_LEN
    assert D == RAW_FEAT_DIM

    finger_chains = [
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16],
        [17, 18, 19, 20],
    ]

    angles_all = []

    for hand_start in [0, 63]:
        hand = X[:, :, hand_start:hand_start+63].reshape(N, T, 21, 3)
        hand_angles = np.zeros((N, T, 15), dtype=np.float32)
        idx = 0

        for chain in finger_chains:
            for i in range(len(chain) - 1):
                a = hand[:, :, (0 if i == 0 else chain[i-1]), :]
                b = hand[:, :, chain[i], :]
                c = hand[:, :, chain[i+1], :]

                v1 = a - b
                v2 = c - b

                denom = (
                    np.linalg.norm(v1, axis=-1) *
                    np.linalg.norm(v2, axis=-1) +
                    1e-8
                )

                cos_a = np.sum(v1 * v2, axis=-1) / denom
                hand_angles[:, :, idx] = np.arccos(np.clip(cos_a, -1, 1))
                idx += 1

        angles_all.append(hand_angles)

    return np.concatenate([X, np.concatenate(angles_all, axis=-1)], axis=-1)


def preprocess_raw_landmarks(X_raw):
    """
    Raw MediaPipe color hand landmarks -> model input.
    """
    X_rel = normalize_hands_relative(X_raw)
    X_feat = compute_finger_angles(X_rel)

    assert X_feat.shape[-1] == FINAL_FEAT_DIM

    X_norm = (X_feat - feat_mean) / feat_std
    X_norm = np.nan_to_num(X_norm, nan=0.0, posinf=0.0, neginf=0.0)

    return X_norm.astype(np.float32)


print("Preprocessing functions ready.")

In [ ]:
# ============================================================
# 11) Test verisini pack et
# ============================================================
RAW_X_PATH = PACKED_DIR / "test_color_final_raw_X.npy"
X_PATH = PACKED_DIR / "test_color_final_X.npy"
Y_ORIG_PATH = PACKED_DIR / "test_color_final_y_original.npy"
Y_NEW_PATH = PACKED_DIR / "test_color_final_y_model.npy"
SID_PATH = PACKED_DIR / "test_color_final_sample_ids.npy"

def build_test_arrays(df):
    X_raw_list = []
    y_orig_list = []
    y_new_list = []
    sample_ids = []
    missing_cache = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Packing test arrays"):
        sid = str(row["sample_id"])
        cache_path = TEST_CACHE / f"{safe_cache_name(sid)}.npy"

        if not cache_path.exists():
            missing_cache += 1
            arr = np.zeros((SEQ_LEN, RAW_FEAT_DIM), dtype=np.float32)
        else:
            arr = np.load(cache_path).astype(np.float32)

        if arr.shape != (SEQ_LEN, RAW_FEAT_DIM):
            print("Bad cached shape:", sid, arr.shape)
            arr = np.zeros((SEQ_LEN, RAW_FEAT_DIM), dtype=np.float32)

        X_raw_list.append(arr)
        sample_ids.append(sid)

        orig_label = int(row["label"])
        y_orig_list.append(orig_label)

        if orig_label >= 0:
            y_new_list.append(orig_to_new[orig_label])
        else:
            y_new_list.append(-1)

    X_raw = np.stack(X_raw_list).astype(np.float32)
    X_model = preprocess_raw_landmarks(X_raw)

    y_orig = np.array(y_orig_list, dtype=np.int32)
    y_new = np.array(y_new_list, dtype=np.int32)
    sample_ids = np.array(sample_ids)

    print("Missing cache:", missing_cache)
    print("X_raw:", X_raw.shape)
    print("X_model:", X_model.shape)
    print("y_orig:", y_orig.shape)
    print("y_new:", y_new.shape)

    np.save(RAW_X_PATH, X_raw)
    np.save(X_PATH, X_model)
    np.save(Y_ORIG_PATH, y_orig)
    np.save(Y_NEW_PATH, y_new)
    np.save(SID_PATH, sample_ids)

    metadata = pd.DataFrame({
        "sample_id": sample_ids,
        "original_label": y_orig,
        "model_label": y_new,
    })
    metadata.to_csv(PACKED_DIR / "test_color_final_metadata.csv", index=False)

    return X_raw, X_model, y_orig, y_new, sample_ids


packed_exists = X_PATH.exists() and Y_NEW_PATH.exists() and SID_PATH.exists()

if packed_exists and not FORCE_REBUILD:
    print("Packed test dosyaları bulundu, tekrar oluşturulmadı.")
    X_test = np.load(X_PATH).astype(np.float32)
    y_test_orig = np.load(Y_ORIG_PATH).astype(np.int32)
    y_test_model = np.load(Y_NEW_PATH).astype(np.int32)
    sample_ids = np.load(SID_PATH, allow_pickle=True).astype(str)

    print("X_test:", X_test.shape)
    print("y_test_orig:", y_test_orig.shape)
    print("y_test_model:", y_test_model.shape)
else:
    X_raw, X_test, y_test_orig, y_test_model, sample_ids = build_test_arrays(known_test_df)

assert X_test.shape[1:] == (SEQ_LEN, FINAL_FEAT_DIM)

In [ ]:
# ============================================================
# 12) Model prediction
# ============================================================
BATCH_SIZE_PRED = 64

proba = model.predict(X_test, batch_size=BATCH_SIZE_PRED, verbose=1)
pred_new = np.argmax(proba, axis=1)
pred_conf = np.max(proba, axis=1)

pred_orig = np.array([new_to_orig[int(i)] for i in pred_new], dtype=np.int32)

print("proba:", proba.shape)
print("pred_new:", pred_new.shape)
print("pred_orig:", pred_orig.shape)
print("confidence mean:", float(pred_conf.mean()))

In [ ]:
# ============================================================
# 13) Prediction CSV kaydet
# ============================================================
def label_info_from_new(new_label):
    info = label_map.get(int(new_label), {})
    return {
        "pred_original_class_id": info.get("original_class_id", new_to_orig.get(int(new_label), -1)),
        "pred_TR": info.get("TR", ""),
        "pred_EN": info.get("EN", ""),
    }

rows = []

top_k = min(5, proba.shape[1])
top_indices = np.argsort(proba, axis=1)[:, -top_k:][:, ::-1]

for i, sid in enumerate(sample_ids):
    pred_info = label_info_from_new(pred_new[i])

    row = {
        "sample_id": sid,
        "true_original_class_id": int(y_test_orig[i]),
        "true_model_label": int(y_test_model[i]),
        "pred_model_label": int(pred_new[i]),
        "pred_original_class_id": int(pred_info["pred_original_class_id"]),
        "pred_TR": pred_info["pred_TR"],
        "pred_EN": pred_info["pred_EN"],
        "confidence": float(pred_conf[i]),
        "correct": bool(y_test_model[i] == pred_new[i]) if y_test_model[i] >= 0 else None,
    }

    for rank in range(top_k):
        cls_new = int(top_indices[i, rank])
        info = label_info_from_new(cls_new)
        row[f"top{rank+1}_model_label"] = cls_new
        row[f"top{rank+1}_original_class_id"] = int(info["pred_original_class_id"])
        row[f"top{rank+1}_TR"] = info["pred_TR"]
        row[f"top{rank+1}_EN"] = info["pred_EN"]
        row[f"top{rank+1}_prob"] = float(proba[i, cls_new])

    rows.append(row)

pred_df = pd.DataFrame(rows)

pred_csv_path = RESULTS_DIR / "test_predictions_final.csv"
pred_df.to_csv(pred_csv_path, index=False, encoding="utf-8-sig")

print("Prediction CSV kaydedildi:")
print(pred_csv_path)
display(pred_df.head(10))

In [ ]:
# ============================================================
# 14) Metrics hesapla
# ============================================================
has_labels = np.all(y_test_model >= 0)

if not has_labels:
    print("Label yok; accuracy hesaplanmadı.")
else:
    num_classes = proba.shape[1]
    labels_all = np.arange(num_classes)

    top1 = accuracy_score(y_test_model, pred_new)
    top3 = top_k_accuracy_score(y_test_model, proba, k=min(3, num_classes), labels=labels_all)
    top5 = top_k_accuracy_score(y_test_model, proba, k=min(5, num_classes), labels=labels_all)

    print("=" * 60)
    print("FINAL MODEL TEST RESULTS")
    print("=" * 60)
    print("Test samples used:", len(y_test_model))
    print("Top-1 accuracy:", f"{top1*100:.2f}%")
    print("Top-3 accuracy:", f"{top3*100:.2f}%")
    print("Top-5 accuracy:", f"{top5*100:.2f}%")
    print("Mean confidence:", f"{pred_conf.mean()*100:.2f}%")
    print("=" * 60)

    metrics = {
        "model_file": str(MODEL_PATH),
        "test_dir": str(TEST_DIR),
        "test_label_file": str(TEST_LABEL_FILE) if TEST_LABEL_FILE is not None else None,
        "num_test_samples_used": int(len(y_test_model)),
        "num_known_model_classes": int(len(label_encoder_classes)),
        "top1_accuracy": float(top1),
        "top3_accuracy": float(top3),
        "top5_accuracy": float(top5),
        "mean_confidence": float(pred_conf.mean()),
        "preprocessing": "color_only + relative_coords + finger_angles + zscore",
        "ignored_files": "*_depth.mp4"
    }

    metrics_path = RESULTS_DIR / "test_metrics_final.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)

    print("\nMetrics JSON kaydedildi:")
    print(metrics_path)

In [ ]:
# ============================================================
# 15) Per-class accuracy
# ============================================================
if has_labels:
    df = pred_df.copy()

    per_class = (
        df.groupby("true_original_class_id")
          .agg(
              n=("correct", "count"),
              correct=("correct", "sum"),
              accuracy=("correct", "mean")
          )
          .reset_index()
          .sort_values(["accuracy", "n"], ascending=[True, False])
    )

    # TR/EN isimlerini ekle
    # label_map yeni label -> original_class_id tuttuğu için ters map kuruyoruz.
    orig_to_name = {}
    for new_label, info in label_map.items():
        orig = int(info.get("original_class_id", new_to_orig.get(int(new_label), -1)))
        orig_to_name[orig] = {
            "TR": info.get("TR", ""),
            "EN": info.get("EN", "")
        }

    per_class["TR"] = per_class["true_original_class_id"].map(lambda x: orig_to_name.get(int(x), {}).get("TR", ""))
    per_class["EN"] = per_class["true_original_class_id"].map(lambda x: orig_to_name.get(int(x), {}).get("EN", ""))
    per_class["accuracy_%"] = per_class["accuracy"] * 100

    per_class_path = RESULTS_DIR / "test_per_class_accuracy_final.csv"
    per_class.to_csv(per_class_path, index=False, encoding="utf-8-sig")

    print("Per-class accuracy kaydedildi:")
    print(per_class_path)

    print("\nEn kötü 20 class:")
    display(per_class.head(20))

    print("\nEn iyi 20 class:")
    display(per_class.tail(20).sort_values(["accuracy", "n"], ascending=[False, False]))

In [ ]:
# ============================================================
# 16) En sık karışan sınıflar
# ============================================================
if has_labels:
    wrong = pred_df[pred_df["correct"] == False].copy()

    if len(wrong) == 0:
        print("Hiç yanlış prediction yok.")
    else:
        confusions = (
            wrong.groupby(["true_original_class_id", "pred_original_class_id"])
                 .size()
                 .reset_index(name="count")
                 .sort_values("count", ascending=False)
        )

        # İsimleri ekle
        confusions["true_TR"] = confusions["true_original_class_id"].map(lambda x: orig_to_name.get(int(x), {}).get("TR", ""))
        confusions["true_EN"] = confusions["true_original_class_id"].map(lambda x: orig_to_name.get(int(x), {}).get("EN", ""))
        confusions["pred_TR"] = confusions["pred_original_class_id"].map(lambda x: orig_to_name.get(int(x), {}).get("TR", ""))
        confusions["pred_EN"] = confusions["pred_original_class_id"].map(lambda x: orig_to_name.get(int(x), {}).get("EN", ""))

        conf_path = RESULTS_DIR / "test_top_confusions_final.csv"
        confusions.to_csv(conf_path, index=False, encoding="utf-8-sig")

        print("Top confusions kaydedildi:")
        print(conf_path)

        display(confusions.head(30))

In [ ]:
# ============================================================
# 17) Confusion matrix kaydet
# ============================================================
if has_labels:
    cm = confusion_matrix(y_test_model, pred_new, labels=np.arange(proba.shape[1]))

    cm_path = RESULTS_DIR / "test_confusion_matrix_final.npy"
    np.save(cm_path, cm)

    print("Confusion matrix kaydedildi:")
    print(cm_path)
    print("Shape:", cm.shape)

In [ ]:
# ============================================================
# 18) Sonuç dosyalarını listele
# ============================================================
print("RESULTS_DIR içeriği:")
for p in sorted(RESULTS_DIR.glob("*")):
    print(" -", p.name)

print("\nPACKED_DIR test dosyaları:")
for p in sorted(PACKED_DIR.glob("test_color_final*")):
    print(" -", p.name)